In [ ]:
!pip install transformers datasets

In [ ]:
!pip install pandas spacy matplotlib seaborn sklearn
!python -m spacy download en_core_web_sm

In [ ]:
from datasets import load_dataset
import pandas as pd

# Load the MeetingBank dataset
dataset = load_dataset("microsoft/MeetingBank-QA-Summary", split="test")
df = pd.DataFrame(dataset)

# Preview the data structure
df[['idx', 'prompt', 'summary', 'gpt4_summary']].head()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Calculate summary length for exploration
df['summary_length'] = df['summary'].apply(lambda x: len(x.split()))

# Visualize distribution of summary lengths
sns.histplot(df['summary_length'], bins=20)
plt.xlabel('Summary Length')
plt.ylabel('Frequency')
plt.title('Distribution of Summary Lengths')
plt.show()

In [ ]:
import spacy

# Load Spacy model
nlp = spacy.load('en_core_web_sm')

# Tokenize and POS tag
def get_pos_tags(text):
    doc = nlp(text)
    return [(token.text, token.pos_) for token in doc]

# Apply POS tagging to a sample prompt
sample_prompt = df['prompt'].iloc[0]
pos_tags = get_pos_tags(sample_prompt)
print(pos_tags)

In [ ]:
# Visualize POS distribution in the sample prompt
pos_counts = pd.Series([pos for _, pos in pos_tags]).value_counts()
pos_counts.plot(kind='bar')
plt.xlabel('POS Tags')
plt.ylabel('Frequency')
plt.title('POS Tag Distribution in Sample Prompt')
plt.show()

In [ ]:
# Named entity recognition
def get_named_entities(text):
    doc = nlp(text)
    return [(ent.text, ent.label_) for ent in doc.ents]

In [ ]:
# Apply NER to a sample prompt
named_entities = get_named_entities(sample_prompt)
print(named_entities)

In [ ]:
# Visualize entity types in the sample prompt
entity_counts = pd.Series([label for _, label in named_entities]).value_counts()
entity_counts.plot(kind='bar')
plt.xlabel('Entity Type')
plt.ylabel('Frequency')
plt.title('Named Entity Distribution in Sample Prompt')
plt.show()

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Convert summaries to TF-IDF matrix
vectorizer = TfidfVectorizer(stop_words='english', max_features=10)
tfidf_matrix = vectorizer.fit_transform(df['summary'])
tfidf_keywords = vectorizer.get_feature_names_out()

print("Top TF-IDF keywords:", tfidf_keywords)

In [ ]:
# Visualize top keywords across summaries
tfidf_sum = tfidf_matrix.sum(axis=0).A1
keyword_counts = pd.Series(tfidf_sum, index=tfidf_keywords)
keyword_counts.plot(kind='bar')
plt.xlabel('Keyword')
plt.ylabel('TF-IDF Score')
plt.title('Top Keywords in Summaries')
plt.show()

In [23]:
import re

# Sample pattern matching to identify topics within summaries
def parse_topics(summary):
    topics = {}
    current_topic = None
    lines = summary.split("\n")

    for line in lines:
        # Check if line is a topic header
        if re.match(r'\[.*\]', line):
            current_topic = line.strip("[]:")
            topics[current_topic] = {"summary": "", "decisions": [], "action_items": []}
        elif current_topic:
            # Check if line contains a decision or action item
            if "Decision:" in line:
                topics[current_topic]["decisions"].append(line.split("Decision:", 1)[1].strip())
            elif "Action Item:" in line:
                topics[current_topic]["action_items"].append(line.split("Action Item:", 1)[1].strip())
            else:
                # Otherwise, add to summary
                topics[current_topic]["summary"] += line.strip() + " "
    return topics

In [ ]:
# Test on a sample summary
sample_summary = df['summary'].iloc[0]
parsed_topics = parse_topics(sample_summary)
print(parsed_topics)

In [ ]:
# Extract sentences with verbs and direct objects as potential action items
def extract_action_items(summary):
    doc = nlp(summary)
    action_items = []
    for token in doc:
        if token.dep_ == "ROOT" and token.pos_ == "VERB":  # Main verbs as roots
            children = [child for child in token.children if child.dep_ == "dobj"]  # Direct objects
            if children:
                action_items.append(token.text + " " + " ".join([child.text for child in children]))
    return action_items


In [ ]:
# Apply on a sample summary
sample_summary = df['summary'].iloc[0]
actions = extract_action_items(sample_summary)
print("Action items:", actions)

In [ ]:
df['summary'].iloc[0]

In [ ]:
import spacy

# Load Spacy model
nlp = spacy.load('en_core_web_sm')

# Function to parse resolutions and implied actions
def extract_resolution(summary):
    doc = nlp(summary)
    resolution = []
    subject = []
    implied_actions = []

    for token in doc:
        # Identify main subjects (e.g., "resolution" or "tenant assessment")
        if token.dep_ in {"nsubj", "ROOT"}:
            subject.append(token.text)

        # Collect actions or resolutions (look for verbs or suggestive phrases)
        if token.pos_ == "VERB" and token.dep_ in {"ROOT", "advcl"}:
            resolution.append(token.text)

        # Implied recommendations or actions from suggestive terms (e.g., "encourage", "use")
        if "encourage" in token.text or "recommend" in token.text or "use" in token.text:
            implied_actions.append(token.text + " " + " ".join([child.text for child in token.children if child.dep_ == "dobj"]))

    return {
        "subject": " ".join(subject).capitalize(),
        "resolution": " ".join(resolution).capitalize(),
        "implied_actions": implied_actions
    }


In [ ]:
# Test the function on a sample summary
sample_summary = df['summary'].iloc[0]
parsed_resolution = extract_resolution(sample_summary)
print("Parsed Resolution:", parsed_resolution)

In [29]:
import json

# Function to convert parsed resolutions into MoM structure
def generate_mom_format(row):
    summary_text = row['summary']
    parsed_resolution = extract_resolution(summary_text)

    # MoM structure
    mom = {
        "Agenda": [parsed_resolution["subject"]],
        "Resolution Summary": {
            parsed_resolution["subject"]: {
                "resolution": parsed_resolution["resolution"],
                "implied_actions": parsed_resolution["implied_actions"]
            }
        },
        "Next Steps": [
            {"description": action, "due_date": "To be decided"} for action in parsed_resolution["implied_actions"]
        ]
    }

    return mom

# Apply on the entire dataset and convert to structured list
mom_data = df.apply(generate_mom_format, axis=1).tolist()

In [ ]:
mom_data

In [ ]:
# Process the dataset for action items
df['action_items'] = df['summary'].apply(extract_action_items)

# Display action items
df[['summary', 'action_items']].head()

In [ ]:
# Visualize action items frequency
action_items_flat = [item for sublist in df['action_items'] for item in sublist]
action_item_counts = pd.Series(action_items_flat).value_counts().head(10)
action_item_counts.plot(kind='bar')
plt.xlabel('Action Item')
plt.ylabel('Frequency')
plt.title('Most Common Action Items')
plt.show()

In [ ]:
import json

# Compile into structured MoM format
def create_mom_format(row):
    return {
        "prompt": row['prompt'],
        "key_points": tfidf_keywords.tolist(),
        "action_items": row['action_items']
    }

In [ ]:
# Convert each row
mom_data = df.apply(create_mom_format, axis=1).tolist()

In [ ]:
mom_data